# 02 - Task Similarity Matrix

This notebook expands the workflow from `examples/compute_task_similarity.py`:

1. build a deterministic multitask model;
2. compute ALE profiles;
3. compare task curves with the Frechet-based similarity helper;
4. inspect nearest-task groups.

In [ ]:
from pathlib import Path
import sys

repo_root = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "pyproject.toml").exists() and (path / "src" / "alemtl").exists()
    ),
    Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve(),
)
for path in (repo_root, repo_root / "src"):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

import torch

from examples.compute_ale_profiles import build_model, make_batches
from alemtl.similarity import MultiTaskALE, MultitaskSimilarity

torch.manual_seed(5)

## ALE Profiles

The deterministic model has three task-specific linear heads. Tasks 0 and 1 are deliberately similar; task 2 is deliberately different.

In [ ]:
model = build_model()
batches = make_batches(n_batches=8, batch_size=48)

ale = MultiTaskALE(
    model=model,
    dataloader=batches,
    n_tasks=3,
    n_features_out=1,
    num_intervals=12,
    n_guess=96,
)
ale.update()
curves = ale(centered=True, cumulative=True, std=1.0)
curves.shape

Curve layout is `(tasks, features, intervals, x_plus_outputs)`. With one output, the last axis is `(x, y)`.

In [ ]:
task = 0
feature = 0
curves[task, feature, :5]

## Pairwise Task Similarity

`MultitaskSimilarity.scores` stores task similarity scores, where larger means more similar.

In [ ]:
similarity = MultitaskSimilarity(ale_curves=ale, centered=True, cumulative=True, std=1.0)
similarity.compute()
similarity.scores

## Feature-level Similarity

The feature tensor keeps one score per `(task, task, feature)`.

In [ ]:
similarity.similarity_tasks_features.shape, similarity.similarity_tasks_features

## Nearest-task Groups

These pairs are what the loss regularizer can consume through `update_tasks_groups`.

In [ ]:
scores, pairs = similarity.tasks_groups()
for score, pair in zip(scores, pairs):
    task, nearest = pair.tolist()
    print(f"task {task} -> task {nearest}, score={score.item():.4f}")